In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

df = pd.read_csv("../data/turbine_hourly.csv", parse_dates=["ts_hour"])
df = df.sort_values(["turbine_id", "ts_hour"])

turbine = "T01"
df_t = df[df["turbine_id"] == turbine].copy()

# filter out downtime for modeling
mask_ok = (df_t["is_downtime"] == 0) & df_t["wind_ms"].notna()
train = df_t[mask_ok]

X = train[["wind_ms", "temp_c"]]
y = train["avg_power_kw"]

model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
model.fit(X, y)

train["pred_power_kw"] = model.predict(X)
print("R^2:", r2_score(y, train["pred_power_kw"]))

# empirical power curve
bins = np.arange(0, 30, 1)
df_t["wind_bin"] = pd.cut(df_t["wind_ms"], bins=bins)
power_curve = df_t.groupby("wind_bin")["avg_power_kw"].mean()

plt.figure(figsize=(6, 4))
power_curve.plot(kind="bar")
plt.title(f"Empirical Power Curve - {turbine}")
plt.ylabel("Average power (kW)")
plt.tight_layout()
plt.show()

# estimate lost energy during downtime
df_t["pred_power_kw"] = model.predict(df_t[["wind_ms", "temp_c"]])
df_t["pred_power_kw"] = df_t["pred_power_kw"].clip(lower=0)

# 1-hour resolution assumed
df_t["expected_energy_kwh"] = df_t["pred_power_kw"] * 1.0
df_t["actual_energy_kwh"] = df_t["energy_kwh"].clip(lower=0)

downtime_mask = df_t["is_downtime"] == 1
df_t["lost_energy_kwh"] = df_t["expected_energy_kwh"] - df_t["actual_energy_kwh"]
df_t.loc[~downtime_mask, "lost_energy_kwh"] = 0

total_lost = df_t["lost_energy_kwh"].sum()
print(f"Estimated lost energy due to downtime for {turbine}: {total_lost:.1f} kWh")

# weather vs downtime
df_t["high_wind_flag"] = (df_t["wind_ms"] > 20).astype(int)
downtime_by_high_wind = df_t.groupby("high_wind_flag")["is_downtime"].mean()
print("Downtime probability by high wind flag:\n", downtime_by_high_wind)

df_t["icing_flag"] = df_t["icing_flag"].astype(int)
downtime_by_icing = df_t.groupby("icing_flag")["is_downtime"].mean()
print("Downtime probability by icing flag:\n", downtime_by_icing)
